# Stage 2 – TF-IDF + linear SVM
**Owner: Samir Karki** · ICT6001 Applied Project

Builds TF-IDF features (unigrams and bigrams, up to 50,000 features, sublinear TF), tunes C on the validation set, runs 5-fold cross-validation and saves the model. The test set is **not** used here; it is used once, in Stage 4.

Runtime: CPU is fine (about 3 minutes). Run Stage 1 first.

In [ ]:
# Setup: connect Google Drive and point to the project folder
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/phishing_project'

import os, sys, json, time, random, pickle, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sys.path.insert(0, PROJECT_DIR)
warnings.filterwarnings('ignore')

SEED = 42
DATA_DIR  = os.path.join(PROJECT_DIR, 'data')
OUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
SPLIT_DIR = os.path.join(OUT_DIR, 'splits')
MODEL_DIR = os.path.join(OUT_DIR, 'models')
FIG_DIR   = os.path.join(OUT_DIR, 'figures')
RUN_LOG   = os.path.join(OUT_DIR, 'run_log.csv')
for d in (OUT_DIR, SPLIT_DIR, MODEL_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style='whitegrid')
C_SVM, C_BERT, C_PHISH, C_LEGIT = '#3B6EA5', '#D9822B', '#C8553D', '#7FB77E'

def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches='tight'); plt.show()
    print('Saved figure:', name)
def save_summary(stage, d):
    with open(os.path.join(OUT_DIR, f'summary_{stage}.json'), 'w') as f: json.dump(d, f, indent=1, default=str)
print('Project folder:', PROJECT_DIR)

## 2.1 Load the splits and build TF-IDF features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score
from src.run_log import log_run

train = pd.read_csv(os.path.join(SPLIT_DIR, 'train.csv')); val = pd.read_csv(os.path.join(SPLIT_DIR, 'val.csv'))
for d in (train, val): d['clean'] = d['clean'].fillna('')
TFIDF = dict(lowercase=True, ngram_range=(1, 2), max_features=50000, sublinear_tf=True, min_df=2, strip_accents='unicode')
start = time.time()
vec = TfidfVectorizer(**TFIDF)
Xtr, Xva = vec.fit_transform(train['clean']), vec.transform(val['clean'])
ytr, yva = train['label'].values, val['label'].values
print('Training matrix:', Xtr.shape, f'| built in {time.time() - start:.1f}s')

## 2.2 Grid search on C (validation set only)

In [ ]:
grid = []
for C in [0.01, 0.1, 1, 10]:
    start = time.time(); m = LinearSVC(C=C, random_state=SEED).fit(Xtr, ytr)
    grid.append({'C': C, 'val_f1': round(f1_score(yva, m.predict(Xva)), 4), 'fit_seconds': round(time.time() - start, 2)})
grid = pd.DataFrame(grid); print(grid)
BEST_C = float(grid.loc[grid['val_f1'].idxmax(), 'C']); print('\nBest C =', BEST_C)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(grid['C'], grid['val_f1'], marker='o', color=C_SVM); ax.set_xscale('log')
ax.set_title('SVM grid search: validation F1 by C'); ax.set_xlabel('C (log scale)'); ax.set_ylabel('Validation F1')
savefig('fig04_svm_grid_search.png')

## 2.3 5-fold stratified cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_f1 = cross_val_score(make_pipeline(TfidfVectorizer(**TFIDF), LinearSVC(C=BEST_C, random_state=SEED)),
                        train['clean'], ytr, cv=cv, scoring='f1', n_jobs=-1)
print('F1 per fold:', np.round(cv_f1, 4)); print(f'Mean {cv_f1.mean():.4f} | std {cv_f1.std():.4f}')

## 2.4 Train the final model and save it

In [ ]:
start = time.time()
svm = LinearSVC(C=BEST_C, random_state=SEED).fit(Xtr, ytr)
fit_seconds = time.time() - start
val_f1 = f1_score(yva, svm.predict(Xva))
with open(os.path.join(MODEL_DIR, 'svm.pkl'), 'wb') as f: pickle.dump({'vectorizer': vec, 'svm': svm}, f)
size_mb = os.path.getsize(os.path.join(MODEL_DIR, 'svm.pkl')) / 1e6
print(f'Final SVM trained in {fit_seconds:.2f}s | validation F1 {val_f1:.4f} | saved model {size_mb:.1f} MB')

## 2.5 Explainability: which words drive the SVM?

In [ ]:
names, coef = np.array(vec.get_feature_names_out()), svm.coef_.ravel()
top_p, top_l = np.argsort(coef)[-15:], np.argsort(coef)[:15]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].barh(names[top_p], coef[top_p], color=C_PHISH); axes[0].set_title('Top 15 phishing indicators')
axes[1].barh(names[top_l][::-1], coef[top_l][::-1], color=C_LEGIT); axes[1].set_title('Top 15 legitimate indicators')
for ax in axes: ax.set_xlabel('SVM weight'); ax.tick_params(axis='y', labelsize=8)
savefig('fig05_svm_top_features.png')
print('Phishing:', list(names[top_p][::-1])); print('Legitimate:', list(names[top_l]))

## 2.6 Log the run

In [ ]:
summary = {'grid': grid.to_dict(orient='records'), 'best_C': BEST_C, 'cv_f1': [round(x, 4) for x in cv_f1],
           'cv_f1_mean': round(cv_f1.mean(), 4), 'cv_f1_std': round(cv_f1.std(), 4), 'val_f1': round(val_f1, 4),
           'fit_seconds': round(fit_seconds, 2), 'size_mb': round(size_mb, 1), 'n_features': int(Xtr.shape[1]),
           'top_phishing_features': list(names[top_p][::-1]), 'top_legitimate_features': list(names[top_l])}
save_summary('02_svm', summary)
log_run(RUN_LOG, 'train', 'TF-IDF + SVM', {**TFIDF, 'C': BEST_C}, {'val_f1': round(val_f1, 4), 'cv_f1_mean': summary['cv_f1_mean']}, 'Samir')
print('Stage 2 complete.')